In [1]:
# Do this only in Colab notebooks! Otherwise use pip install unsloth
import os, re
import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
!pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth
!pip install transformers==4.57.3
!pip install --no-deps trl==0.22.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 11.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 9.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 40.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depen

In [ ]:
from unsloth import FastLanguageModel
import torch

print("Loading LFM2.5-1.2B-Instruct from LiquidAI...")
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name="unsloth/LFM2.5-1.2B-Instruct",
    model_name="unsloth/LFM2.5-1.2B-Instruct",
    max_seq_length=8192,  # Choose any for long context!
    load_in_4bit=True,  # 4 bit quantization to reduce memory
    load_in_8bit=False,  # [NEW!] A bit more accurate, uses 2x memory
    load_in_16bit=False,  # [NEW!] Enables 16bit LoRA
    full_finetuning=False,  # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)
print("Model loaded successfully!")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj",
                      "w1", "w2", "w3"],
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
    loftq_config = None,
)

In [2]:
import ast
from datasets import load_dataset, concatenate_datasets
from unsloth.chat_templates import get_chat_template, standardize_data_formats

# 1. Enforce a standard Chat Template on your tokenizer
# Unsloth automatically maps this to your specific base model's required format
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", 
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-03-16 21:55:46.886507: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773698147.094830      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773698147.159606      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773698147.691616      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773698147.691654      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773698147.691657      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


NameError: name 'tokenizer' is not defined

### Load 2 datasets - Hinglish, English Pro

In [3]:
print("Loading gooftagoo dataset...")
# Gooftagoo has ~16.2k rows. We sample ,000.
gooftagoo_ds = load_dataset("adi-kmt/gooftagoo", split="train")
# gooftagoo_ds = gooftagoo_ds.select(range(20000))

print("Loading English anchor dataset...")
# FineTome acts as our logic anchor. We sample 2,000.
english_ds = load_dataset("mlabonne/FineTome-100k", split="train")
english_ds = english_ds.select(range(2000))

Loading gooftagoo dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16207 [00:00<?, ? examples/s]

Loading English anchor dataset...


README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [5]:
def format_gooftagoo(example):
    # Safely evaluate the string into a Python list
    try:
        raw_turns = ast.literal_eval(example["conversation"])
    except (ValueError, SyntaxError):
        raw_turns = []
        
    new_conversations = []
    # Break apart the {'user': '...', 'assistant': '...'} pairs into standard turns
    for turn in raw_turns:
        if "user" in turn:
            new_conversations.append({"role": "user", "content": turn["user"]})
        if "assistant" in turn:
            new_conversations.append({"role": "assistant", "content": turn["assistant"]})
            
    return {"conversations": new_conversations}

In [6]:
def format_english(example):
    new_conversations = []
    for msg in example["conversations"]:
        # Handle if it's already in role/content format
        if "role" in msg and "content" in msg:
            role = msg["role"]
            content = msg["content"]
        # Handle if it's in ShareGPT from/value format (just to be safe)
        elif "from" in msg and "value" in msg:
            role = msg["from"]
            if role == "human": role = "user"
            elif role == "gpt": role = "assistant"
            content = msg["value"]
        else:
            continue
            
        new_conversations.append({"role": role, "content": content})
        
    return {"conversations": new_conversations}

In [7]:
print("Standardizing Gooftagoo...")
gooftagoo_ds = gooftagoo_ds.map(format_gooftagoo)

print("Standardizing English dataset...")
english_ds = english_ds.map(format_english)

Standardizing Gooftagoo...


Map:   0%|          | 0/16207 [00:00<?, ? examples/s]

Standardizing English dataset...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [8]:
gooftagoo_ds[0]

{'conversation': "[{'user': ' ठंड बहुत है, हाथ जल्दी से गरम करने का कोई तरीका बता ना।', 'assistant': 'अरे, हाथ रगड़ने से गरम हो जाएंगे। लेकिन बता, तेरे हाथ अभी कैसे हैं? सूखे हैं, गीले हैं या कुछ लगा रखा है?'}, {'user': 'अभी तो सूखे हैं, लेकिन मैं सोच रहा था कि तेल या लोशन लगा लूं।', 'assistant': 'नहीं नहीं, अगर तुझे जल्दी से हाथ गरम करने हैं, तो सूखे हाथों से रगड़ना सबसे बेहतर होगा। गीले हाथों से या तेल, लोशन लगे हाथों से रगड़ने पर उतनी गर्मी नहीं मिलेगी।'}, {'user': 'अच्छा, सूखे हाथों से ही रगड़ना है। लेकिन ऐसा क्यों?', 'assistant': 'देख, जब हम सूखे हाथ रगड़ते हैं, तो घर्षण ज्यादा होता है, जिससे ज्यादा गर्मी पैदा होती है। गीले हाथों में या तेल, लोशन लगे हाथों में घर्षण कम होता है, इसलिए गर्मी कम मिलती है।'}]",
 'messages': [{'content': 'ठंड बहुत है, हाथ जल्दी से गरम करने का कोई तरीका बता ना।',
   'role': 'user'},
  {'content': 'अरे, हाथ रगड़ने से गरम हो जाएंगे। लेकिन बता, तेरे हाथ अभी कैसे हैं? सूखे हैं, गीले हैं या कुछ लगा रखा है?',
   'role': 'assistant'},
  {'content': 'अभी तो सूख

In [9]:
english_ds[20]

{'conversations': [{'content': 'Write a program to find the number of letters in each word of a sentence using the map function.',
   'role': 'user'},
  {'content': 'Certainly! Here\'s a code example that uses the `map()` function to find the number of letters in each word of a sentence:\n\n```python\ndef count_letters(sentence):\n    """\n    This function takes a sentence as input and returns a list of the number of letters\n    in each word using the map function.\n\n    Args:\n        sentence (str): The input sentence containing words.\n\n    Returns:\n        list: A list of integers representing the number of letters in each word.\n\n    Examples:\n        >>> count_letters("Hello world")\n        [5, 5]\n        >>> count_letters("Python is awesome")\n        [6, 2, 7]\n    """\n    # Split the sentence into words\n    words = sentence.split()\n\n    # Use map to apply len() function on each word and get the length\n    letter_counts = list(map(len, words))\n\n    return letter

In [10]:
def unify_schema(example):
    standardized_turns = []

    # We check 'conversations' first, fallback to 'messages' if needed
    chat_data = example.get("conversations", example.get("messages", []))

    for msg in chat_data:
        # Catch both OpenAI (role/content) and ShareGPT (from/value) formats just in case
        role = msg.get("role", msg.get("from", ""))
        content = msg.get("content", msg.get("value", ""))

        # Standardize ShareGPT roles if they exist
        if role == "human": role = "user"
        if role == "gpt": role = "assistant"

        standardized_turns.append({
            "role": str(role),
            "content": str(content)
        })

    return {"conversations": standardized_turns}

In [11]:
gooftagoo_ds = gooftagoo_ds.map(unify_schema, remove_columns=gooftagoo_ds.column_names)
english_ds = english_ds.map(unify_schema, remove_columns=english_ds.column_names)

Map:   0%|          | 0/16207 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [12]:
gooftagoo_ds[0]["conversations"]

[{'content': ' ठंड बहुत है, हाथ जल्दी से गरम करने का कोई तरीका बता ना।',
  'role': 'user'},
 {'content': 'अरे, हाथ रगड़ने से गरम हो जाएंगे। लेकिन बता, तेरे हाथ अभी कैसे हैं? सूखे हैं, गीले हैं या कुछ लगा रखा है?',
  'role': 'assistant'},
 {'content': 'अभी तो सूखे हैं, लेकिन मैं सोच रहा था कि तेल या लोशन लगा लूं।',
  'role': 'user'},
 {'content': 'नहीं नहीं, अगर तुझे जल्दी से हाथ गरम करने हैं, तो सूखे हाथों से रगड़ना सबसे बेहतर होगा। गीले हाथों से या तेल, लोशन लगे हाथों से रगड़ने पर उतनी गर्मी नहीं मिलेगी।',
  'role': 'assistant'},
 {'content': 'अच्छा, सूखे हाथों से ही रगड़ना है। लेकिन ऐसा क्यों?',
  'role': 'user'},
 {'content': 'देख, जब हम सूखे हाथ रगड़ते हैं, तो घर्षण ज्यादा होता है, जिससे ज्यादा गर्मी पैदा होती है। गीले हाथों में या तेल, लोशन लगे हाथों में घर्षण कम होता है, इसलिए गर्मी कम मिलती है।',
  'role': 'assistant'}]

In [13]:
english_ds[20]["conversations"]

[{'content': 'Write a program to find the number of letters in each word of a sentence using the map function.',
  'role': 'user'},
 {'content': 'Certainly! Here\'s a code example that uses the `map()` function to find the number of letters in each word of a sentence:\n\n```python\ndef count_letters(sentence):\n    """\n    This function takes a sentence as input and returns a list of the number of letters\n    in each word using the map function.\n\n    Args:\n        sentence (str): The input sentence containing words.\n\n    Returns:\n        list: A list of integers representing the number of letters in each word.\n\n    Examples:\n        >>> count_letters("Hello world")\n        [5, 5]\n        >>> count_letters("Python is awesome")\n        [6, 2, 7]\n    """\n    # Split the sentence into words\n    words = sentence.split()\n\n    # Use map to apply len() function on each word and get the length\n    letter_counts = list(map(len, words))\n\n    return letter_counts\n```\n\nNow,

In [14]:
gooftagoo_ds.filter(lambda x: len(x["conversations"]) < 0)

Filter:   0%|          | 0/16207 [00:00<?, ? examples/s]

Dataset({
    features: ['conversations'],
    num_rows: 0
})

In [15]:
english_ds.filter(lambda x: len(x["conversations"]) < 0)

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['conversations'],
    num_rows: 0
})

In [16]:
mixed_dataset = concatenate_datasets([gooftagoo_ds, english_ds])
mixed_dataset = mixed_dataset.shuffle(seed=3407)

In [17]:
len(mixed_dataset)

18207

In [18]:
def formatting_prompts_func(examples):
    texts = tokenizer.apply_chat_template(
        examples["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    # Remove BOS token if Unsloth/Tokenizer adds it automatically to prevent double-tokens
    return {"text": [x.removeprefix(tokenizer.bos_token) for x in texts]}

In [ ]:
mixed_dataset = mixed_dataset.map(formatting_prompts_func, batched=True)

In [ ]:
mixed_dataset[0]["text"]

In [ ]:
print("Splitting into Train and Eval sets...")
split_dataset = mixed_dataset.train_test_split(test_size=0.1, seed=3407)
train_data = split_dataset["train"]
eval_data = split_dataset["test"]

print(f"Final Training Rows: {len(train_data)} | Evaluation Rows: {len(eval_data)}")

In [ ]:
from trl import SFTTrainer, SFTConfig
# from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 1. Initialize the Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    eval_dataset = eval_data, # We pass the eval set here so we can watch validation loss!
    # max_seq_length = 2048, # Capped at 2k context for training to protect 6GB VRAM
    # dataset_num_proc = 2,
    neftune_noise_alpha=5,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, # Lowest possible batch size
        gradient_accumulation_steps = 8, # Simulates a batch size of 8

        num_train_epochs = 1,
        # max_steps=60,

        # Evaluation Settings
        logging_steps = 1,
        eval_strategy = "steps",
        eval_steps = 10, # Prints validation loss at every 10th step

        learning_rate = 2e-4,
        warmup_steps = 30,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim = "paged_adamw_8bit", # The VRAM safety net
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,

        output_dir = "outputs",
        report_to = "none", # Change to "wandb" if you set up a Weights & Biases account
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
print("Exporting model to Q4_K_M GGUF format for llama.cpp...")
# This merges your LoRA weights with the base model and quantizes it to 4-bit
model.save_pretrained_gguf("hinglish_1.5B_instruct_gguf", tokenizer, quantization_method="q4_k_m")

print("Pipeline complete! Your Hinglish model is ready.")

In [19]:
from transformers import AutoTokenizer
from unsloth.chat_templates import get_chat_template

print("Loading global tokenizer for dataset prep...")
# Load ONLY the tokenizer on the CPU. This is perfectly safe!
global_tokenizer = AutoTokenizer.from_pretrained("unsloth/LFM2.5-1.2B-Instruct")
global_tokenizer = get_chat_template(global_tokenizer, chat_template="chatml")

def formatting_prompts_func(examples):
    texts = global_tokenizer.apply_chat_template(
        examples["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    # Remove BOS token if Unsloth/Tokenizer adds it automatically to prevent double-tokens
    return {"text": [x.removeprefix(global_tokenizer.bos_token) for x in texts]}

print("Applying chat template to dataset...")
mixed_dataset = mixed_dataset.map(formatting_prompts_func, batched=True)

print("Splitting into Train and Eval sets...")
split_dataset = mixed_dataset.train_test_split(test_size=0.1, seed=3407)
train_data = split_dataset["train"]
eval_data = split_dataset["test"]

print(f"Ready! Training Rows: {len(train_data)} | Evaluation Rows: {len(eval_data)}")

Loading global tokenizer for dataset prep...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


Applying chat template to dataset...


Map:   0%|          | 0/18207 [00:00<?, ? examples/s]

Splitting into Train and Eval sets...
Ready! Training Rows: 16386 | Evaluation Rows: 1821


In [20]:
from accelerate import notebook_launcher
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig

# 1. Define the training loop function
def train_loop():
    # Import inside the function is safer for multi-processing
    from unsloth import FastLanguageModel
    
    print("Loading model on parallel processes...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/LFM2.5-1.2B-Instruct",
        max_seq_length=8192,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=64,
        target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"],
        lora_alpha=128,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=True,
    )

    # 1. Enforce a standard Chat Template on your tokenizer
    # Unsloth automatically maps this to your specific base model's required format
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = "chatml", 
    )

    pritn("Setting up trainer")
    # 2. Setup Trainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_data,  # This pulls from your globally defined dataset
        eval_dataset=eval_data,
        neftune_noise_alpha=5,
        args=SFTConfig(
            dataset_text_field="text",
            
            # MULTI-GPU MATH:
            per_device_train_batch_size=1, # 1 per GPU = 2 simultaneous 
            gradient_accumulation_steps=4, # 2 (GPUs) * 1 (bs) * 4 (grad_acc) = 8 total effective batch size
            num_train_epochs=1,            # Your full epoch!
            
            # EVALUATION FIX:
            eval_strategy="steps",
            eval_steps=200,                # Evaluates every ~200 steps instead of every 1 step
            logging_steps=10,
            
            learning_rate=2e-4,
            warmup_steps=30,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            optim="paged_adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            seed=3407,
            output_dir="outputs",
            report_to="none",
            ddp_find_unused_parameters=False, # Recommended for PEFT/LoRA in DDP
        ),
    )
    
    print("Applying response-only training")
    # 3. Apply response-only training
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )

    print("Started training")
    # 4. Start training
    trainer.train()

    # 5. SAVE THE LORA ADAPTERS! (Add this right at the end of the function)
    print("Saving the model to disk...")
    # trainer.save_model is safe for multi-gpu, it only saves on the main process
    trainer.save_model("outputs/unsloth_lora_model")
    tokenizer.save_pretrained("outputs/unsloth_lora_model")
    print("Training and saving complete!")

In [21]:
# 5. Launch the distributed training!
notebook_launcher(train_loop, num_processes=2)

RuntimeError: Could not start distributed process. Libraries known to initialize device upon import have been imported already. Please keep these imports inside your training function to try and help with this:
	* `bitsandbytes`

In [1]:
import ast
from datasets import load_dataset, concatenate_datasets
from accelerate import notebook_launcher

def train_loop():
    print("========== SCRIPT INITIATED ==========")
    print("--- Starting Step 1: Quarantined Imports ---")
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from unsloth.chat_templates import get_chat_template, train_on_responses_only
    from trl import SFTTrainer, SFTConfig
    print("--- Finished Step 1: Quarantined Imports ---")
    
    print("--- Starting Step 2: Loading Model & Tokenizer ---")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/LFM2.5-1.2B-Instruct",
        max_seq_length=8192,
        load_in_4bit=True,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="chatml")
    print("--- Finished Step 2: Loading Model & Tokenizer ---")

    print("--- Starting Step 3: Loading Datasets ---")
    # All rows for Gooftagoo, 2000 rows for English FineTome
    gooftagoo_ds = load_dataset("adi-kmt/gooftagoo", split="train")
    english_ds = load_dataset("mlabonne/FineTome-100k", split="train").select(range(2000))
    print(f"Loaded Gooftagoo: {len(gooftagoo_ds)} rows")
    print(f"Loaded English: {len(english_ds)} rows")
    print("--- Finished Step 3: Loading Datasets ---")

    print("--- Starting Step 4: Applying Custom Formatting Functions ---")
    def format_english(example):
        new_conversations = []
        for msg in example["conversations"]:
            # Handle if it's already in role/content format
            if "role" in msg and "content" in msg:
                role = msg["role"]
                content = msg["content"]
            # Handle if it's in ShareGPT from/value format (just to be safe)
            elif "from" in msg and "value" in msg:
                role = msg["from"]
                if role == "human": role = "user"
                elif role == "gpt": role = "assistant"
                content = msg["value"]
            else:
                continue
            new_conversations.append({"role": role, "content": content})
        return {"conversations": new_conversations}

    def format_gooftagoo(example):
        # Safely evaluate the string into a Python list
        try:
            raw_turns = ast.literal_eval(example["conversation"])
        except (ValueError, SyntaxError):
            raw_turns = []
            
        new_conversations = []
        # Break apart the {'user': '...', 'assistant': '...'} pairs into standard turns
        for turn in raw_turns:
            if "user" in turn:
                new_conversations.append({"role": "user", "content": turn["user"]})
            if "assistant" in turn:
                new_conversations.append({"role": "assistant", "content": turn["assistant"]})
                
        return {"conversations": new_conversations}

    # Apply the mapping and remove old columns to prevent schema conflicts during concatenation
    gooftagoo_ds = gooftagoo_ds.map(format_gooftagoo, remove_columns=gooftagoo_ds.column_names)
    english_ds = english_ds.map(format_english, remove_columns=english_ds.column_names)
    print("--- Finished Step 4: Applying Custom Formatting Functions ---")

    print("--- Starting Step 5: Concatenating and Applying Chat Template ---")
    mixed_dataset = concatenate_datasets([gooftagoo_ds, english_ds]).shuffle(seed=3407)

    def format_with_template(examples):
        texts = tokenizer.apply_chat_template(examples["conversations"], tokenize=False, add_generation_prompt=False)
        return {"text": [x.removeprefix(tokenizer.bos_token) for x in texts]}

    mixed_dataset = mixed_dataset.map(format_with_template, batched=True)
    print("--- Finished Step 5: Concatenating and Applying Chat Template ---")

    print("--- Starting Step 6: Splitting Dataset ---")
    split_dataset = mixed_dataset.train_test_split(test_size=0.1, seed=3407)
    train_data = split_dataset["train"]
    eval_data = split_dataset["test"]
    print(f"Final Train Rows: {len(train_data)} | Final Eval Rows: {len(eval_data)}")
    print("--- Finished Step 6: Splitting Dataset ---")

    print("--- Starting Step 7: Applying LoRA Adapters ---")
    model = FastLanguageModel.get_peft_model(
        model,
        r=64,
        target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"],
        lora_alpha=128,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=True,
    )
    print("--- Finished Step 7: Applying LoRA Adapters ---")

    print("--- Starting Step 8: Setting Up SFTTrainer ---")
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_data,  
        eval_dataset=eval_data,
        neftune_noise_alpha=5,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=1, 
            gradient_accumulation_steps=4, 
            num_train_epochs=1,            
            eval_strategy="steps",
            eval_steps=200,                
            logging_steps=10,
            learning_rate=2e-4,
            warmup_steps=30,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            optim="paged_adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            seed=3407,
            output_dir="outputs",
            report_to="none",
            ddp_find_unused_parameters=False,
        ),
    )

    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    print("--- Finished Step 8: Setting Up SFTTrainer ---")

    print("--- Starting Step 9: Training Model ---")
    trainer.train()
    print("--- Finished Step 9: Training Model ---")

    print("--- Starting Step 10: Saving LoRA Adapters ---")
    trainer.save_model("outputs/unsloth_lora_model")
    tokenizer.save_pretrained("outputs/unsloth_lora_model")
    print("--- Finished Step 10: Saving LoRA Adapters ---")
    print("========== SCRIPT COMPLETE ==========")

In [ ]:
# Launch on both T4 GPUs
notebook_launcher(train_loop, num_processes=2)

In [2]:
%%writefile train_multi_gpu.py
import ast
from datasets import load_dataset, concatenate_datasets
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTTrainer, SFTConfig

print("========== SCRIPT INITIATED ==========")

print("--- Starting Step 1: Loading Model & Tokenizer ---")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/LFM2.5-1.2B-Instruct",
    max_seq_length=8192,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")
print("--- Finished Step 1: Loading Model & Tokenizer ---")

print("--- Starting Step 2: Loading Datasets ---")
gooftagoo_ds = load_dataset("adi-kmt/gooftagoo", split="train")
english_ds = load_dataset("mlabonne/FineTome-100k", split="train").select(range(2000))
print(f"Loaded Gooftagoo: {len(gooftagoo_ds)} rows")
print(f"Loaded English: {len(english_ds)} rows")
print("--- Finished Step 2: Loading Datasets ---")

print("--- Starting Step 3: Applying Custom Formatting Functions ---")
def format_english(example):
    new_conversations = []
    for msg in example["conversations"]:
        if "role" in msg and "content" in msg:
            role = msg["role"]
            content = msg["content"]
        elif "from" in msg and "value" in msg:
            role = msg["from"]
            if role == "human": role = "user"
            elif role == "gpt": role = "assistant"
            content = msg["value"]
        else:
            continue
        new_conversations.append({"role": role, "content": content})
    return {"conversations": new_conversations}

def format_gooftagoo(example):
    try:
        raw_turns = ast.literal_eval(example["conversation"])
    except (ValueError, SyntaxError):
        raw_turns = []
        
    new_conversations = []
    for turn in raw_turns:
        if "user" in turn:
            new_conversations.append({"role": "user", "content": turn["user"]})
        if "assistant" in turn:
            new_conversations.append({"role": "assistant", "content": turn["assistant"]})
            
    return {"conversations": new_conversations}

gooftagoo_ds = gooftagoo_ds.map(format_gooftagoo, remove_columns=gooftagoo_ds.column_names)
english_ds = english_ds.map(format_english, remove_columns=english_ds.column_names)
print("--- Finished Step 3: Applying Custom Formatting Functions ---")

print("--- Starting Step 4: Concatenating and Applying Chat Template ---")
mixed_dataset = concatenate_datasets([gooftagoo_ds, english_ds]).shuffle(seed=3407)

def format_with_template(examples):
    texts = tokenizer.apply_chat_template(examples["conversations"], tokenize=False, add_generation_prompt=False)
    return {"text": [x.removeprefix(tokenizer.bos_token) for x in texts]}

mixed_dataset = mixed_dataset.map(format_with_template, batched=True)
print("--- Finished Step 4: Concatenating and Applying Chat Template ---")

print("--- Starting Step 5: Splitting Dataset ---")
split_dataset = mixed_dataset.train_test_split(test_size=0.1, seed=3407)
train_data = split_dataset["train"]
eval_data = split_dataset["test"]
print(f"Final Train Rows: {len(train_data)} | Final Eval Rows: {len(eval_data)}")
print("--- Finished Step 5: Splitting Dataset ---")

print("--- Starting Step 6: Applying LoRA Adapters ---")
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=True,
)
print("--- Finished Step 6: Applying LoRA Adapters ---")

print("--- Starting Step 7: Setting Up SFTTrainer ---")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,  
    eval_dataset=eval_data,
    neftune_noise_alpha=5,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1, 
        gradient_accumulation_steps=4, 
        num_train_epochs=1,            
        eval_strategy="steps",
        eval_steps=200,                
        logging_steps=10,
        learning_rate=2e-4,
        warmup_steps=30,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        ddp_find_unused_parameters=False,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
print("--- Finished Step 7: Setting Up SFTTrainer ---")

print("--- Starting Step 8: Training Model ---")
trainer.train()
print("--- Finished Step 8: Training Model ---")

print("--- Starting Step 9: Saving LoRA Adapters ---")
trainer.save_model("outputs/unsloth_lora_model")
tokenizer.save_pretrained("outputs/unsloth_lora_model")
print("--- Finished Step 9: Saving LoRA Adapters ---")
print("========== SCRIPT COMPLETE ==========")

Writing train_multi_gpu.py


In [3]:
!accelerate launch --multi_gpu --num_processes=2 train_multi_gpu.py

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-03-16 22:13:58.782506: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-16 22:13:58.782573: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773699239.224247     166 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attemp

In [10]:
from unsloth import FastLanguageModel
import torch

print("Loading the 500-step checkpoint...")
# Notice we point model_name directly to your checkpoint folder!
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/working/outputs/checkpoint-500", 
    max_seq_length=4196,
    load_in_4bit=True,
)

# Crucial: This optimizes the model for 2x faster inference
FastLanguageModel.for_inference(model) 

# Let's set up a bilingual prompt to test it!
messages = [
    {"role": "user", "content": "Hello! I want to learn machine learning. कहाँ से शुरू करूँ?"}
]

# Apply the ChatML template
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True, # Tells the model it's its turn to speak
    return_tensors="pt",
).to("cuda")

print("Generating response...")
# Generate the output
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True) # skip_prompt hides the template tags from output

print("\n--- MODEL RESPONSE ---\n")
_ = model.generate(
    input_ids=inputs, 
    streamer = text_streamer, 
    max_new_tokens = 256, 
    use_cache = True,
    temperature = 0.4,
    repetition_penalty = 1.15,
    no_repeat_ngram_size = 3, #
)

Loading the 500-step checkpoint...
==((====))==  Unsloth 2026.3.4: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Generating response...

--- MODEL RESPONSE ---

Bilkul, Machine Learning is a fascinating field that has many applications in various fields like healthcare, education, and technology. Here are some basic concepts you should understand:

1. **Data Types**:
   - There are three main types of data: numerical, categorical, and textual.
   - Numeric Data: Numbers can be represented as integers or floating-point numbers.
   Categorical Data: This type of data represents distinct categories such as colors, lang

In [11]:
ROOT_PATH = "my_model"
os.makedirs(ROOT_PATH, exist_ok=True)
model.save_pretrained(f"{ROOT_PATH}/lora_r16_4bit_model")  # Local saving
tokenizer.save_pretrained(f"{ROOT_PATH}/lora_r16_4bit_model")

('my_model/lora_r16_4bit_model/tokenizer_config.json',
 'my_model/lora_r16_4bit_model/special_tokens_map.json',
 'my_model/lora_r16_4bit_model/chat_template.jinja',
 'my_model/lora_r16_4bit_model/tokenizer.json')

In [12]:
model.save_pretrained_gguf(
    f"{ROOT_PATH}/sft_multilingual_hindi",
    tokenizer,
    quantization_method = ["q4_k_m", "q8_0", "f16"], 
)

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `my_model/sft_multilingual_hindi`: 100%|██████████| 1/1 [00:01<00:00,  1.28s/it]


Successfully copied all 1 files from cache to `my_model/sft_multilingual_hindi`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:16<00:00, 16.69s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/my_model/sft_multilingual_hindi`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m', 'q8_0', 'f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['my_model/sft_multilingual_hindi_gguf/LFM2.5-1.2B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: [2] Converting GGUF f16 into q8_0. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['my_model/sft_multilingual_hindi_gguf/LFM2.5-1.2B-Instruct.Q8_0.gguf', 'my_model/sft_multilingual_hindi_gguf/LFM2.5-1.2B-Instruct.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/LFM2.5-1.2B-Instruct'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model my_model/sft_multilingual_hindi_gguf/LFM2.5-1.2B-Instruct.Q8_0.gguf -p "why is the sky blue?"


{'save_directory': 'my_model/sft_multilingual_hindi',
 'gguf_directory': 'my_model/sft_multilingual_hindi_gguf',
 'gguf_files': ['my_model/sft_multilingual_hindi_gguf/LFM2.5-1.2B-Instruct.Q8_0.gguf',
  'my_model/sft_multilingual_hindi_gguf/LFM2.5-1.2B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': True,
 'is_vlm': False,
 'fix_bos_token': False}

In [13]:
%%writefile train_multi_gpu2.py
import ast
from datasets import load_dataset, concatenate_datasets
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTTrainer, SFTConfig

print("========== SCRIPT INITIATED ==========")

print("--- Starting Step 1: Loading Model & Tokenizer ---")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/LFM2.5-1.2B-Instruct",
    max_seq_length=4096,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")
print("--- Finished Step 1: Loading Model & Tokenizer ---")

print("--- Starting Step 2: Loading Datasets ---")
gooftagoo_ds = load_dataset("adi-kmt/gooftagoo", split="train")
english_ds = load_dataset("mlabonne/FineTome-100k", split="train").select(range(2000))
print(f"Loaded Gooftagoo: {len(gooftagoo_ds)} rows")
print(f"Loaded English: {len(english_ds)} rows")
print("--- Finished Step 2: Loading Datasets ---")

print("--- Starting Step 3: Applying Custom Formatting Functions ---")
def format_english(example):
    new_conversations = []
    for msg in example["conversations"]:
        if "role" in msg and "content" in msg:
            role = msg["role"]
            content = msg["content"]
        elif "from" in msg and "value" in msg:
            role = msg["from"]
            if role == "human": role = "user"
            elif role == "gpt": role = "assistant"
            content = msg["value"]
        else:
            continue
        new_conversations.append({"role": role, "content": content})
    return {"conversations": new_conversations}

def format_gooftagoo(example):
    try:
        raw_turns = ast.literal_eval(example["conversation"])
    except (ValueError, SyntaxError):
        raw_turns = []
        
    new_conversations = []
    for turn in raw_turns:
        if "user" in turn:
            new_conversations.append({"role": "user", "content": turn["user"]})
        if "assistant" in turn:
            new_conversations.append({"role": "assistant", "content": turn["assistant"]})
            
    return {"conversations": new_conversations}

gooftagoo_ds = gooftagoo_ds.map(format_gooftagoo, remove_columns=gooftagoo_ds.column_names)
english_ds = english_ds.map(format_english, remove_columns=english_ds.column_names)
print("--- Finished Step 3: Applying Custom Formatting Functions ---")

print("--- Starting Step 4: Concatenating and Applying Chat Template ---")
mixed_dataset = concatenate_datasets([gooftagoo_ds, english_ds]).shuffle(seed=3407)

def format_with_template(examples):
    texts = tokenizer.apply_chat_template(examples["conversations"], tokenize=False, add_generation_prompt=False)
    return {"text": [x.removeprefix(tokenizer.bos_token) for x in texts]}

mixed_dataset = mixed_dataset.map(format_with_template, batched=True)
print("--- Finished Step 4: Concatenating and Applying Chat Template ---")

print("--- Starting Step 5: Splitting Dataset ---")
split_dataset = mixed_dataset.train_test_split(test_size=0.1, seed=3407)
train_data = split_dataset["train"]
eval_data = split_dataset["test"]
print(f"Final Train Rows: {len(train_data)} | Final Eval Rows: {len(eval_data)}")
print("--- Finished Step 5: Splitting Dataset ---")

print("--- Starting Step 6: Applying LoRA Adapters ---")
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=True,
)
print("--- Finished Step 6: Applying LoRA Adapters ---")

print("--- Starting Step 7: Setting Up SFTTrainer ---")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,  
    eval_dataset=eval_data,
    neftune_noise_alpha=5,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1, 
        gradient_accumulation_steps=4, 
        num_train_epochs=1,            
        eval_strategy="steps",
        eval_steps=200,                
        logging_steps=10,
        learning_rate=2e-4,
        warmup_steps=30,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        ddp_find_unused_parameters=False,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
print("--- Finished Step 7: Setting Up SFTTrainer ---")

print("--- Starting Step 8: Resuming Training from Checkpoint ---")
# This tells the trainer to pick up exactly at step 500!
trainer.train(resume_from_checkpoint="outputs/checkpoint-500")
print("--- Finished Step 8: Training Model ---")

print("--- Starting Step 9: Saving LoRA Adapters ---")
trainer.save_model("outputs/unsloth_lora_model")
tokenizer.save_pretrained("outputs/unsloth_lora_model")
print("--- Finished Step 9: Saving LoRA Adapters ---")
print("========== SCRIPT COMPLETE ==========")

Writing train_multi_gpu2.py


In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True accelerate launch --multi_gpu --num_processes=2 train_multi_gpu2.py

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-03-16 23:56:05.235254: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-16 23:56:05.235299: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773705365.257805   10094 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attemp